# All comparisons of TDC tasks

In [20]:
from pathlib import Path
import json
from collections import defaultdict
import pandas as pd
import numpy as np
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)


In [21]:
# Define paths
OUR_RESULTS_DIR = Path("../../data/results/tdc_tasks")
OUR_RESULTS_DIR_NO_DESC = Path("../../data/results/tdc_tasks_no_description")
PMO_10K_RESULTS_DIR = Path("../../data/results/pmo_baseline")
PMO_50_RESULTS_CSV = Path("../../data/results/tdc_top_50_baselines/results.csv")

# Define model mappings from filename to display name (PMO baselines)
MODEL_MAPPING_10K = {
    'reinvent': 'REINVENT',
    'reinvent_selfies': 'REINVENT SELFIES',
    'graph_ga': 'Graph GA',
    'gp_bo': 'GP BO',
    'stoned': 'STONED',
}

MODEL_MAPPING_50 = {
    'reinvent': 'REINVENT',
    'reinvent_selfies': 'REINVENT SELFIES',
    'graph_ga': 'Graph GA',
    'gpbo': 'GP BO',
}

# Our model names
OUR_MODEL_NAME = 'LLM Agent (Ours)'
OUR_MODEL_NAME_NO_DESC = 'LLM Agent (No Description)'

# List of all 23 PMO benchmark tasks
TASKS = [
    'albuterol_similarity',
    'amlodipine_mpo',
    'celecoxib_rediscovery',
    'deco_hop',
    'drd2',
    'fexofenadine_mpo',
    'gsk3b',
    'isomers_c7h8n2o2',
    'isomers_c9h10n2o2pf2cl',
    'jnk3',
    'median1',
    'median2',
    'mestranol_similarity',
    'osimertinib_mpo',
    'perindopril_mpo',
    'qed',
    'ranolazine_mpo',
    'scaffold_hop',
    'sitagliptin_mpo',
    'thiothixene_rediscovery',
    'troglitazone_rediscovery',
    'valsartan_smarts',
    'zaleplon_mpo',
]

## Loading our data (LLM Agent)

In [22]:
def load_trace_and_config(p: Path) -> tuple[list, dict]:
    """Load trace and config from JSON file."""
    d = json.loads(p.read_text(encoding="utf-8"))
    trace = d["trace"] if isinstance(d, dict) and "trace" in d else d
    config = d.get("config", {}) if isinstance(d, dict) else {}
    return trace, config


def load_our_results_as_df(results_dir: Path, model_name: str = None) -> pd.DataFrame:
    """Load all our results from JSON trace files into a DataFrame.
    
    Fills in missing iterations (due to invalid SMILES) with score=0 to ensure
    consistent oracle call counts across all runs.
    
    Args:
        results_dir: Directory containing task subdirectories with JSON files
        model_name: Name to use for the 'model' column. If None, uses OUR_MODEL_NAME.
    
    Returns DataFrame with columns: model, task, run, score, oracle_call
    """
    if model_name is None:
        model_name = OUR_MODEL_NAME
    
    records = []
    
    for task_dir in results_dir.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        for run_idx, json_file in enumerate(sorted(task_dir.glob("*.json"))):
            try:
                trace, config = load_trace_and_config(json_file)
                
                # Get max_iterations from config, or infer from trace
                max_iterations = config.get("objective", {}).get("params", {}).get("max_iterations")
                if max_iterations is None:
                    # Fall back to max iteration in trace
                    max_iterations = max(entry["iteration"] for entry in trace) if trace else 0
                
                # Build a map of iteration -> score from the trace
                iteration_to_score = {int(entry["iteration"]): float(entry["score"]) for entry in trace}
                
                # Create records for all iterations from 1 to max_iterations
                # Missing iterations (invalid SMILES) get score=0
                for iteration in range(1, max_iterations):
                    records.append({
                        'model': model_name,
                        'task': task_name,
                        'run': run_idx,
                        'score': iteration_to_score.get(iteration, 0.0),
                        'oracle_call': iteration
                    })
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
    
    return pd.DataFrame(records)


# Load our results (with description)
our_df = load_our_results_as_df(OUR_RESULTS_DIR, OUR_MODEL_NAME)
print(f"Our results (with description): {len(our_df)} rows, {our_df['task'].nunique()} tasks, {our_df.groupby('task')['run'].nunique().max()} max runs")

# Load our results (no description)
our_df_no_desc = load_our_results_as_df(OUR_RESULTS_DIR_NO_DESC, OUR_MODEL_NAME_NO_DESC)
print(f"Our results (no description): {len(our_df_no_desc)} rows, {our_df_no_desc['task'].nunique()} tasks, {our_df_no_desc.groupby('task')['run'].nunique().max()} max runs")

# Combine both our results
our_df = pd.concat([our_df, our_df_no_desc], ignore_index=True)
print(f"\nCombined our results: {len(our_df)} rows, {our_df['model'].nunique()} model variants")
our_df.head()

Our results (with description): 4200 rows, 28 tasks, 3 max runs
Our results (no description): 3450 rows, 23 tasks, 3 max runs

Combined our results: 7650 rows, 2 model variants


,model,task,run,score,oracle_call
0,LLM Agent (Ours),deco_hop,0,0.870448,1
1,LLM Agent (Ours),deco_hop,0,0.834904,2
2,LLM Agent (Ours),deco_hop,0,0.841983,3
3,LLM Agent (Ours),deco_hop,0,0.870448,4
4,LLM Agent (Ours),deco_hop,0,0.860705,5


## Loading PMO 10K results (for AUC over 10000 iterations)

In [23]:
def load_yaml_results_fast(filepath):
    """Load results from a YAML file using fast custom parsing."""
    results = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    i = 0
    n = len(lines)
    while i < n:
        line = lines[i]
        if not line.strip():
            i += 1
            continue
        if not line.startswith('-'):
            if i + 2 < n:
                score_line = lines[i + 1].strip()
                oracle_line = lines[i + 2].strip()
                if score_line.startswith('- ') and oracle_line.startswith('- '):
                    try:
                        score = float(score_line[2:])
                        oracle_call = int(oracle_line[2:])
                        results.append((score, oracle_call))
                    except (ValueError, IndexError):
                        pass
                    i += 3
                    continue
        i += 1
    
    results.sort(key=lambda x: x[1])
    return results


def load_pmo_10k_results_as_df(results_dir: Path, model_mapping: dict) -> pd.DataFrame:
    """Load raw PMO results from YAML files into a DataFrame."""
    records = []
    
    for model_key, model_name in model_mapping.items():
        for task in TASKS:
            pattern = f"results_{model_key}_{task}_*.yaml"
            for run_idx, filepath in enumerate(sorted(results_dir.glob(pattern))):
                results = load_yaml_results_fast(filepath)
                for score, oracle_call in results:
                    records.append({
                        'model': model_name,
                        'task': task,
                        'run': run_idx,
                        'score': score,
                        'oracle_call': oracle_call
                    })
    
    return pd.DataFrame(records)


print("Loading PMO baseline results (10K oracle calls) from YAML files...")
pmo_10k_df = load_pmo_10k_results_as_df(PMO_10K_RESULTS_DIR, MODEL_MAPPING_10K)
print(f"PMO 10K results: {len(pmo_10k_df)} rows, {pmo_10k_df['model'].nunique()} models")
print(f"Runs per model: {pmo_10k_df.groupby('model')['run'].nunique().to_dict()}")
pmo_10k_df.head()

Loading PMO baseline results (10K oracle calls) from YAML files...
PMO 10K results: 4848166 rows, 5 models
Runs per model: {'GP BO': 5, 'Graph GA': 5, 'REINVENT': 5, 'REINVENT SELFIES': 5, 'STONED': 5}


,model,task,run,score,oracle_call
0,REINVENT,albuterol_similarity,0,0.237624,1
1,REINVENT,albuterol_similarity,0,0.289855,2
2,REINVENT,albuterol_similarity,0,0.291188,3
3,REINVENT,albuterol_similarity,0,0.233333,4
4,REINVENT,albuterol_similarity,0,0.241135,5


## Loading PMO 50-iteration results (for AUC over 50 iterations)

In [24]:
def load_pmo_50_results_as_df(csv_path: Path, model_mapping: dict) -> pd.DataFrame:
    """Load PMO 50-iteration results from CSV into a DataFrame.
    
    CSV columns: model, task, repeat, iteration, molecule, score, rank, duration_seconds, timestamp
    """
    df = pd.read_csv(csv_path)
    
    # Rename columns to match our format
    df = df.rename(columns={
        'repeat': 'run',
        'iteration': 'oracle_call'
    })
    
    # Map model names
    df['model'] = df['model'].map(model_mapping)
    
    # Keep only relevant columns
    df = df[['model', 'task', 'run', 'score', 'oracle_call']].copy()
    
    # Drop rows with unmapped models
    df = df.dropna(subset=['model'])
    
    return df


print("Loading PMO baseline results (50 iterations) from CSV...")
pmo_50_df = load_pmo_50_results_as_df(PMO_50_RESULTS_CSV, MODEL_MAPPING_50)
print(f"PMO 50 results: {len(pmo_50_df)} rows, {pmo_50_df['model'].nunique()} models")
print(f"Tasks covered: {sorted(pmo_50_df['task'].unique())}")
pmo_50_df.head()

Loading PMO baseline results (50 iterations) from CSV...
PMO 50 results: 23460 rows, 4 models
Tasks covered: ['albuterol_similarity', 'amlodipine_mpo', 'celecoxib_rediscovery', 'deco_hop', 'drd2', 'fexofenadine_mpo', 'gsk3b', 'isomers_c7h8n2o2', 'isomers_c9h10n2o2pf2cl', 'jnk3', 'median1', 'median2', 'mestranol_similarity', 'osimertinib_mpo', 'perindopril_mpo', 'qed', 'ranolazine_mpo', 'scaffold_hop', 'sitagliptin_mpo', 'thiothixene_rediscovery', 'troglitazone_rediscovery', 'valsartan_smarts', 'zaleplon_mpo']


,model,task,run,score,oracle_call
0,REINVENT,qed,0,0.904836,6
1,REINVENT,qed,0,0.888410,18
2,REINVENT,qed,0,0.886922,45
3,REINVENT,qed,0,0.885346,51
4,REINVENT,qed,0,0.880454,47


---
## AUC Top-1 over 10000 iterations

In [25]:
def compute_auc_top1_from_df(group_df, max_oracle_calls=10000):
    """Compute AUC of top-1 (best) score vs oracle calls from a dataframe group.
    
    Uses trapezoidal integration (same as PMO benchmark).
    
    Args:
        group_df: DataFrame with 'score' and 'oracle_call' columns for a single run
        max_oracle_calls: Maximum number of oracle calls (default 10000)
    
    Returns:
        AUC value normalized to [0, 1]
    """
    sorted_df = group_df.sort_values('oracle_call')
    
    best_score = 0.0
    best_at_call = {}
    
    for _, row in sorted_df.iterrows():
        oracle_call = int(row['oracle_call'])
        score = float(row['score'])
        
        if oracle_call > max_oracle_calls:
            break
        
        if score > best_score:
            best_score = score
        
        best_at_call[oracle_call] = best_score
    
    if not best_at_call:
        return 0.0
    
    oracle_calls = sorted(best_at_call.keys())
    
    # Compute AUC using trapezoidal integration (same as PMO benchmark)
    auc = 0.0
    prev_call = 0
    prev_best = 0.0
    
    for call in oracle_calls:
        # Trapezoidal rule: area = width * (height1 + height2) / 2
        auc += (call - prev_call) * (best_at_call[call] + prev_best) / 2
        prev_call = call
        prev_best = best_at_call[call]
    
    # Extend to max_oracle_calls with the last value
    auc += (max_oracle_calls - prev_call) * prev_best
    
    return auc / max_oracle_calls


def bootstrap_sum_ci(df, metric_col, n_bootstrap=10000, ci=0.95, seed=42):
    """
    Bootstrap confidence interval for sum across tasks.
    
    For each bootstrap iteration:
    - For each task, randomly sample one run (with replacement)
    - Sum the metric across all tasks
    
    Args:
        df: DataFrame with columns ['model', 'task', 'run', metric_col]
        metric_col: Name of the metric column (e.g., 'auc_top1' or 'best_score')
        n_bootstrap: Number of bootstrap iterations
        ci: Confidence interval level (default 0.95 for 95% CI)
        seed: Random seed for reproducibility
    
    Returns:
        DataFrame with columns ['model', 'sum_mean', 'sum_ci_lower', 'sum_ci_upper']
    """
    np.random.seed(seed)
    results = []
    
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        tasks = model_df['task'].unique()
        
        # Build a dict: task -> list of metric values (one per run)
        task_values = {}
        for task in tasks:
            task_df = model_df[model_df['task'] == task]
            task_values[task] = task_df[metric_col].values
        
        # Bootstrap
        boot_sums = []
        for _ in range(n_bootstrap):
            boot_sum = 0
            for task in tasks:
                # Sample one run's value for this task (with replacement)
                sampled_value = np.random.choice(task_values[task])
                boot_sum += sampled_value
            boot_sums.append(boot_sum)
        
        boot_sums = np.array(boot_sums)
        
        # Compute percentiles for CI
        alpha = 1 - ci
        ci_lower = np.percentile(boot_sums, alpha / 2 * 100)
        ci_upper = np.percentile(boot_sums, (1 - alpha / 2) * 100)
        
        results.append({
            'model': model,
            'sum_mean': np.mean(boot_sums),
            'sum_ci_lower': ci_lower,
            'sum_ci_upper': ci_upper
        })
    
    return pd.DataFrame(results)


def create_comparison_table(auc_summary, model_sums_ci, tasks, model_order):
    """Create a comparison table DataFrame from AUC summary.
    
    Args:
        auc_summary: DataFrame with per-task mean/std
        model_sums_ci: DataFrame with bootstrap CI for sums (from bootstrap_sum_ci)
        tasks: List of task names
        model_order: List of model names in display order
    """
    rows = []
    for task in tasks:
        row = {'Task': task}
        task_data = auc_summary[auc_summary['task'] == task]
        
        for model in model_order:
            model_data = task_data[task_data['model'] == model]
            if len(model_data) > 0:
                mean_val = model_data['auc_mean'].values[0]
                std_val = model_data['auc_std'].values[0]
                if pd.isna(std_val) or std_val == 0:
                    row[model] = f"{mean_val:.3f}"
                else:
                    row[model] = f"{mean_val:.3f}± {std_val:.3f}"
            else:
                row[model] = "-"
        rows.append(row)
    
    # Add Sum row with bootstrap 95% CI (as relative values)
    sum_row = {'Task': 'Sum'}
    for model in model_order:
        model_ci = model_sums_ci[model_sums_ci['model'] == model]
        if len(model_ci) > 0:
            mean_val = model_ci['sum_mean'].values[0]
            ci_lower = model_ci['sum_ci_lower'].values[0]
            ci_upper = model_ci['sum_ci_upper'].values[0]
            ci_upper_rel = ci_upper - mean_val
            ci_lower_rel = mean_val - ci_lower
            sum_row[model] = f"{mean_val:.3f} [+{ci_upper_rel:.3f}, -{ci_lower_rel:.3f}]"
        else:
            sum_row[model] = "-"
    rows.append(sum_row)
    
    # Add Rank row
    rank_row = {'Task': 'Rank'}
    for i, model in enumerate(model_order):
        rank_row[model] = str(i + 1)
    rows.append(rank_row)
    
    table = pd.DataFrame(rows)
    return table[['Task'] + model_order]

In [26]:
# Combine our results (both variants) with PMO 10K results
df_10k = pd.concat([our_df, pmo_10k_df], ignore_index=True)
print(f"Combined 10K dataframe: {len(df_10k)} rows, {df_10k['model'].nunique()} models")

# Compute AUC Top-1 for each model/task/run
MAX_ORACLE_CALLS = 10000

auc_records = []
for (model, task, run), group in df_10k.groupby(['model', 'task', 'run']):
    auc = compute_auc_top1_from_df(group, MAX_ORACLE_CALLS)
    auc_records.append({
        'model': model,
        'task': task,
        'run': run,
        'auc_top1': auc
    })

auc_df = pd.DataFrame(auc_records)
print(f"Computed AUC Top-1 for {len(auc_df)} model/task/run combinations")

# Compute mean and std AUC per model/task
auc_summary = auc_df.groupby(['model', 'task'])['auc_top1'].agg(['mean', 'std']).reset_index()
auc_summary.columns = ['model', 'task', 'auc_mean', 'auc_std']

# Compute bootstrap 95% CI for sum across tasks
print("Computing bootstrap 95% CI for sums (10K iterations)...")
model_sums_ci = bootstrap_sum_ci(auc_df, 'auc_top1', n_bootstrap=10000, ci=0.95, seed=42)
model_sums_ci = model_sums_ci.sort_values('sum_mean', ascending=False).reset_index(drop=True)
model_sums_ci['rank'] = range(1, len(model_sums_ci) + 1)

# Display with relative CI values
model_sums_ci_display = model_sums_ci.copy()
model_sums_ci_display['ci_upper'] = model_sums_ci_display['sum_ci_upper'] - model_sums_ci_display['sum_mean']
model_sums_ci_display['ci_lower'] = model_sums_ci_display['sum_mean'] - model_sums_ci_display['sum_ci_lower']
model_sums_ci_display = model_sums_ci_display[['model', 'sum_mean', 'ci_upper', 'ci_lower', 'rank']]

print(f"\nTop models by sum of AUC Top-1 (10K iterations) with 95% CI:")
print(model_sums_ci_display.to_string(index=False, formatters={
    'sum_mean': '{:.3f}'.format,
    'ci_upper': '+{:.3f}'.format,
    'ci_lower': '-{:.3f}'.format
}))

Combined 10K dataframe: 4855816 rows, 7 models
Computed AUC Top-1 for 728 model/task/run combinations
Computing bootstrap 95% CI for sums (10K iterations)...

Top models by sum of AUC Top-1 (10K iterations) with 95% CI:
                     model sum_mean ci_upper ci_lower  rank
          LLM Agent (Ours)   21.173   +0.586   -0.559     1
                  REINVENT   14.741   +0.946   -0.504     2
                  Graph GA   14.383   +0.447   -0.475     3
          REINVENT SELFIES   14.104   +0.333   -0.328     4
                     GP BO   13.832   +0.598   -0.596     5
                    STONED   13.289   +0.462   -0.396     6
LLM Agent (No Description)   10.443   +0.911   -1.096     7


In [27]:
# Create comparison table for 10K iterations
model_order = model_sums_ci['model'].tolist()
comparison_table_10k = create_comparison_table(auc_summary, model_sums_ci, TASKS, model_order)

print("AUC Top-1 over 10000 iterations:")
comparison_table_10k

AUC Top-1 over 10000 iterations:


,Task,LLM Agent (Ours),REINVENT,Graph GA,REINVENT SELFIES,GP BO,STONED,LLM Agent (No Description)
0,albuterol_similarity,1.000,0.906± 0.004,0.877± 0.025,0.855± 0.036,0.925± 0.013,0.756± 0.087,0.537± 0.107
1,amlodipine_mpo,0.905± 0.001,0.655± 0.042,0.688± 0.023,0.628± 0.023,0.609± 0.049,0.618± 0.054,0.532± 0.032
2,celecoxib_rediscovery,1.000,0.803± 0.110,0.684± 0.137,0.618± 0.044,0.809± 0.084,0.389± 0.050,0.444± 0.072
3,deco_hop,0.961± 0.022,0.682± 0.054,0.627± 0.006,0.649± 0.024,0.648± 0.030,0.616± 0.010,0.598± 0.003
4,drd2,1.000± 0.000,0.968± 0.008,0.991± 0.002,0.979± 0.004,0.958± 0.008,0.935± 0.022,0.673± 0.558
5,fexofenadine_mpo,0.992± 0.008,0.804± 0.008,0.777± 0.013,0.765± 0.005,0.743± 0.008,0.806± 0.020,0.611± 0.048
6,gsk3b,0.983± 0.029,0.894± 0.050,0.829± 0.078,0.824± 0.039,0.879± 0.044,0.704± 0.062,0.482± 0.123
7,isomers_c7h8n2o2,1.000,0.884± 0.033,0.899± 0.067,0.890± 0.039,0.747± 0.125,0.914± 0.011,0.860± 0.128
8,isomers_c9h10n2o2pf2cl,1.000± 0.000,0.673± 0.066,0.766± 0.052,0.781± 0.027,0.514± 0.192,0.823± 0.032,0.499± 0.175
9,jnk3,0.503± 0.075,0.814± 0.028,0.598± 0.159,0.671± 0.077,0.593± 0.179,0.543± 0.105,0.163± 0.097


---
## AUC Top-1 over 50 iterations

In [28]:
# Combine our results (both variants) with PMO 50 results
df_50 = pd.concat([our_df, pmo_50_df], ignore_index=True)
print(f"Combined 50-iter dataframe: {len(df_50)} rows, {df_50['model'].nunique()} models")

# Compute AUC Top-1 for each model/task/run with 50 iterations
MAX_ORACLE_CALLS_50 = 50

auc_records_50 = []
for (model, task, run), group in df_50.groupby(['model', 'task', 'run']):
    auc = compute_auc_top1_from_df(group, MAX_ORACLE_CALLS_50)
    auc_records_50.append({
        'model': model,
        'task': task,
        'run': run,
        'auc_top1': auc
    })

auc_df_50 = pd.DataFrame(auc_records_50)
print(f"Computed AUC Top-1 (50 iterations) for {len(auc_df_50)} model/task/run combinations")

# Compute mean and std AUC per model/task
auc_summary_50 = auc_df_50.groupby(['model', 'task'])['auc_top1'].agg(['mean', 'std']).reset_index()
auc_summary_50.columns = ['model', 'task', 'auc_mean', 'auc_std']

# Compute bootstrap 95% CI for sum across tasks
print("Computing bootstrap 95% CI for sums (50 iterations)...")
model_sums_ci_50 = bootstrap_sum_ci(auc_df_50, 'auc_top1', n_bootstrap=10000, ci=0.95, seed=42)
model_sums_ci_50 = model_sums_ci_50.sort_values('sum_mean', ascending=False).reset_index(drop=True)
model_sums_ci_50['rank'] = range(1, len(model_sums_ci_50) + 1)

# Display with relative CI values
model_sums_ci_50_display = model_sums_ci_50.copy()
model_sums_ci_50_display['ci_upper'] = model_sums_ci_50_display['sum_ci_upper'] - model_sums_ci_50_display['sum_mean']
model_sums_ci_50_display['ci_lower'] = model_sums_ci_50_display['sum_mean'] - model_sums_ci_50_display['sum_ci_lower']
model_sums_ci_50_display = model_sums_ci_50_display[['model', 'sum_mean', 'ci_upper', 'ci_lower', 'rank']]

print(f"\nTop models by sum of AUC Top-1 (50 iterations) with 95% CI:")
print(model_sums_ci_50_display.to_string(index=False, formatters={
    'sum_mean': '{:.3f}'.format,
    'ci_upper': '+{:.3f}'.format,
    'ci_lower': '-{:.3f}'.format
}))

Combined 50-iter dataframe: 31110 rows, 6 models
Computed AUC Top-1 (50 iterations) for 613 model/task/run combinations
Computing bootstrap 95% CI for sums (50 iterations)...

Top models by sum of AUC Top-1 (50 iterations) with 95% CI:
                     model sum_mean ci_upper ci_lower  rank
          LLM Agent (Ours)   19.852   +0.409   -0.383     1
LLM Agent (No Description)    8.501   +0.829   -0.780     2
                  REINVENT    7.064   +0.512   -0.428     3
                  Graph GA    7.041   +0.486   -0.416     4
                     GP BO    6.949   +0.341   -0.306     5
          REINVENT SELFIES    6.881   +0.386   -0.354     6


In [29]:
# Create comparison table for 50 iterations
model_order_50 = model_sums_ci_50['model'].tolist()
comparison_table_50 = create_comparison_table(auc_summary_50, model_sums_ci_50, TASKS, model_order_50)

print("AUC Top-1 over 50 iterations:")
comparison_table_50

AUC Top-1 over 50 iterations:


,Task,LLM Agent (Ours),LLM Agent (No Description),REINVENT,Graph GA,GP BO,REINVENT SELFIES
0,albuterol_similarity,0.990,0.407± 0.062,0.358± 0.021,0.384± 0.026,0.372± 0.028,0.388± 0.014
1,amlodipine_mpo,0.882± 0.001,0.465± 0.009,0.432± 0.033,0.425± 0.019,0.430± 0.035,0.436± 0.024
2,celecoxib_rediscovery,0.990,0.386± 0.025,0.262± 0.036,0.258± 0.021,0.250± 0.013,0.232± 0.023
3,deco_hop,0.934± 0.017,0.568± 0.002,0.558± 0.008,0.801± 0.009,0.803± 0.008,0.796± 0.011
4,drd2,0.990± 0.000,0.432± 0.418,0.195± 0.166,0.168± 0.108,0.050± 0.029,0.167± 0.115
5,fexofenadine_mpo,0.971± 0.006,0.521± 0.032,0.573± 0.022,0.510± 0.026,0.562± 0.008,0.560± 0.062
6,gsk3b,0.965± 0.030,0.314± 0.148,0.141± 0.023,0.134± 0.045,0.163± 0.040,0.121± 0.020
7,isomers_c7h8n2o2,0.990,0.760± 0.068,0.078± 0.099,0.045± 0.067,0.055± 0.087,0.050± 0.038
8,isomers_c9h10n2o2pf2cl,0.985± 0.002,0.383± 0.144,0.257± 0.077,0.229± 0.178,0.122± 0.038,0.234± 0.107
9,jnk3,0.429± 0.039,0.091± 0.048,0.075± 0.022,0.083± 0.023,0.072± 0.034,0.062± 0.008


---
## Best Score after 50 iterations

In [30]:
def compute_best_score(group_df, max_oracle_calls=50):
    """Compute the best score achieved within max_oracle_calls."""
    filtered = group_df[group_df['oracle_call'] <= max_oracle_calls]
    if len(filtered) == 0:
        return 0.0
    return filtered['score'].max()


# Compute best score for each model/task/run
MAX_ITERATIONS_BEST = 50

best_score_records = []
for (model, task, run), group in df_50.groupby(['model', 'task', 'run']):
    best = compute_best_score(group, MAX_ITERATIONS_BEST)
    best_score_records.append({
        'model': model,
        'task': task,
        'run': run,
        'best_score': best
    })

best_score_df = pd.DataFrame(best_score_records)
print(f"Computed best score (50 iterations) for {len(best_score_df)} model/task/run combinations")

# Compute mean and std best score per model/task
best_score_summary = best_score_df.groupby(['model', 'task'])['best_score'].agg(['mean', 'std']).reset_index()
best_score_summary.columns = ['model', 'task', 'score_mean', 'score_std']

# Compute bootstrap 95% CI for sum across tasks
print("Computing bootstrap 95% CI for sums (Best Score)...")
model_sums_ci_best = bootstrap_sum_ci(best_score_df, 'best_score', n_bootstrap=10000, ci=0.95, seed=42)
model_sums_ci_best = model_sums_ci_best.sort_values('sum_mean', ascending=False).reset_index(drop=True)
model_sums_ci_best['rank'] = range(1, len(model_sums_ci_best) + 1)

# Display with relative CI values
model_sums_ci_best_display = model_sums_ci_best.copy()
model_sums_ci_best_display['ci_upper'] = model_sums_ci_best_display['sum_ci_upper'] - model_sums_ci_best_display['sum_mean']
model_sums_ci_best_display['ci_lower'] = model_sums_ci_best_display['sum_mean'] - model_sums_ci_best_display['sum_ci_lower']
model_sums_ci_best_display = model_sums_ci_best_display[['model', 'sum_mean', 'ci_upper', 'ci_lower', 'rank']]

print(f"\nTop models by sum of Best Score (50 iterations) with 95% CI:")
print(model_sums_ci_best_display.to_string(index=False, formatters={
    'sum_mean': '{:.3f}'.format,
    'ci_upper': '+{:.3f}'.format,
    'ci_lower': '-{:.3f}'.format
}))

Computed best score (50 iterations) for 613 model/task/run combinations
Computing bootstrap 95% CI for sums (Best Score)...

Top models by sum of Best Score (50 iterations) with 95% CI:
                     model sum_mean ci_upper ci_lower  rank
          LLM Agent (Ours)   21.179   +0.588   -0.561     1
LLM Agent (No Description)   10.453   +0.912   -1.098     2
                  Graph GA    8.181   +0.553   -0.481     3
                  REINVENT    8.064   +0.881   -0.618     4
                     GP BO    8.003   +0.529   -0.480     5
          REINVENT SELFIES    7.927   +0.659   -0.576     6


In [31]:
# Create comparison table for best score
def create_best_score_table(score_summary, model_sums_ci, tasks, model_order):
    """Create a comparison table DataFrame from best score summary.
    
    Args:
        score_summary: DataFrame with per-task mean/std
        model_sums_ci: DataFrame with bootstrap CI for sums (from bootstrap_sum_ci)
        tasks: List of task names
        model_order: List of model names in display order
    """
    rows = []
    for task in tasks:
        row = {'Task': task}
        task_data = score_summary[score_summary['task'] == task]
        
        for model in model_order:
            model_data = task_data[task_data['model'] == model]
            if len(model_data) > 0:
                mean_val = model_data['score_mean'].values[0]
                std_val = model_data['score_std'].values[0]
                if pd.isna(std_val) or std_val == 0:
                    row[model] = f"{mean_val:.3f}"
                else:
                    row[model] = f"{mean_val:.3f}± {std_val:.3f}"
            else:
                row[model] = "-"
        rows.append(row)
    
    # Add Sum row with bootstrap 95% CI (as relative values)
    sum_row = {'Task': 'Sum'}
    for model in model_order:
        model_ci = model_sums_ci[model_sums_ci['model'] == model]
        if len(model_ci) > 0:
            mean_val = model_ci['sum_mean'].values[0]
            ci_lower = model_ci['sum_ci_lower'].values[0]
            ci_upper = model_ci['sum_ci_upper'].values[0]
            ci_upper_rel = ci_upper - mean_val
            ci_lower_rel = mean_val - ci_lower
            sum_row[model] = f"{mean_val:.3f} [+{ci_upper_rel:.3f}, -{ci_lower_rel:.3f}]"
        else:
            sum_row[model] = "-"
    rows.append(sum_row)
    
    # Add Rank row
    rank_row = {'Task': 'Rank'}
    for i, model in enumerate(model_order):
        rank_row[model] = str(i + 1)
    rows.append(rank_row)
    
    table = pd.DataFrame(rows)
    return table[['Task'] + model_order]


model_order_best = model_sums_ci_best['model'].tolist()
comparison_table_best = create_best_score_table(best_score_summary, model_sums_ci_best, TASKS, model_order_best)

print("Best Score after 50 iterations:")
comparison_table_best

Best Score after 50 iterations:


,Task,LLM Agent (Ours),LLM Agent (No Description),Graph GA,REINVENT,GP BO,REINVENT SELFIES
0,albuterol_similarity,1.000,0.538± 0.108,0.422± 0.034,0.390± 0.028,0.412± 0.039,0.401± 0.019
1,amlodipine_mpo,0.905± 0.001,0.532± 0.032,0.463± 0.020,0.471± 0.030,0.464± 0.028,0.475± 0.030
2,celecoxib_rediscovery,1.000,0.444± 0.073,0.293± 0.033,0.297± 0.046,0.285± 0.033,0.250± 0.034
3,deco_hop,0.961± 0.022,0.598± 0.003,0.818± 0.008,0.572± 0.012,0.820± 0.011,0.811± 0.012
4,drd2,1.000± 0.000,0.675± 0.559,0.249± 0.150,0.322± 0.307,0.072± 0.045,0.244± 0.171
5,fexofenadine_mpo,0.992± 0.008,0.611± 0.048,0.579± 0.031,0.627± 0.042,0.627± 0.027,0.642± 0.034
6,gsk3b,0.983± 0.029,0.483± 0.123,0.212± 0.093,0.204± 0.106,0.206± 0.058,0.158± 0.051
7,isomers_c7h8n2o2,1.000,0.860± 0.128,0.087± 0.103,0.149± 0.227,0.075± 0.093,0.131± 0.115
8,isomers_c9h10n2o2pf2cl,1.000,0.500± 0.175,0.422± 0.162,0.411± 0.077,0.354± 0.181,0.354± 0.201
9,jnk3,0.503± 0.075,0.163± 0.097,0.110± 0.019,0.128± 0.058,0.170± 0.137,0.158± 0.141


---
## AUC Top-10 over 1000 iterations

This metric computes the AUC of the **average of top 10 unique molecules** (by score) seen so far at each iteration, integrated over 1000 oracle calls.

For our model (which runs for 50 iterations), we keep the top-10 average constant for iterations 51-1000.

Uniqueness is verified using RDKit canonical SMILES.

In [32]:
from rdkit import Chem

def canonicalize_smiles(smiles: str) -> str | None:
    """Convert SMILES to canonical form using RDKit. Returns None if invalid."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None


def load_our_results_with_smiles(results_dir: Path, model_name: str = None) -> pd.DataFrame:
    """Load all our results from JSON trace files into a DataFrame, including SMILES.
    
    Args:
        results_dir: Directory containing task subdirectories with JSON files
        model_name: Name to use for the 'model' column.
    
    Returns DataFrame with columns: model, task, run, score, oracle_call, smiles
    """
    if model_name is None:
        model_name = OUR_MODEL_NAME
    
    records = []
    
    for task_dir in results_dir.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        for run_idx, json_file in enumerate(sorted(task_dir.glob("*.json"))):
            try:
                trace, config = load_trace_and_config(json_file)
                
                for entry in trace:
                    records.append({
                        'model': model_name,
                        'task': task_name,
                        'run': run_idx,
                        'score': float(entry["score"]),
                        'oracle_call': int(entry["iteration"]),
                        'smiles': entry.get("smiles", "")
                    })
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
    
    return pd.DataFrame(records)


def load_pmo_10k_results_with_smiles(results_dir: Path, model_mapping: dict) -> pd.DataFrame:
    """Load raw PMO results from YAML files into a DataFrame, including SMILES."""
    records = []
    
    for model_key, model_name in model_mapping.items():
        for task in TASKS:
            pattern = f"results_{model_key}_{task}_*.yaml"
            for run_idx, filepath in enumerate(sorted(results_dir.glob(pattern))):
                # Parse YAML file manually for speed
                with open(filepath, 'r') as f:
                    lines = f.readlines()
                
                i = 0
                n = len(lines)
                while i < n:
                    line = lines[i].strip()
                    if line and not line.startswith('-'):
                        # This is a SMILES line
                        smiles = line.rstrip(':')
                        if i + 2 < n:
                            score_line = lines[i + 1].strip()
                            oracle_line = lines[i + 2].strip()
                            if score_line.startswith('- ') and oracle_line.startswith('- '):
                                try:
                                    score = float(score_line[2:])
                                    oracle_call = int(oracle_line[2:])
                                    records.append({
                                        'model': model_name,
                                        'task': task,
                                        'run': run_idx,
                                        'score': score,
                                        'oracle_call': oracle_call,
                                        'smiles': smiles
                                    })
                                except (ValueError, IndexError):
                                    pass
                            i += 3
                            continue
                    i += 1
    
    return pd.DataFrame(records)


import heapq

def compute_auc_top10_from_df(group_df, max_oracle_calls=1000, check_uniqueness=True, top_n=10):
    """Compute AUC of top-n average score vs oracle calls from a dataframe group.
    
    Uses trapezoidal integration (same as PMO benchmark).
    At each oracle call, we compute the average of the top n unique molecules seen so far.
    If fewer than n molecules are available, averages whatever is available.
    
    Uses a min-heap for O(log n) updates instead of sorting all molecules each time.
    
    Args:
        group_df: DataFrame with 'score', 'oracle_call', 'smiles' columns for a single run
        max_oracle_calls: Maximum number of oracle calls for AUC (default 1000)
        check_uniqueness: If True, use RDKit canonical SMILES to check uniqueness.
                         If False, assume molecules are already unique (for PMO baselines).
        top_n: Number of top molecules to average (default 10)
    
    Returns:
        AUC value normalized to [0, 1]
    """
    sorted_df = group_df.sort_values('oracle_call')
    
    # Track all unique molecules and their best scores: key -> score
    unique_molecules = {}
    # Min-heap of (score, key) for top-n tracking - smallest score at top
    top_n_heap = []
    # Set of keys currently in the heap
    top_n_keys = set()
    # Running sum of top-n scores for fast average calculation
    top_n_sum = 0.0
    
    top_n_avg_at_call = {}
    
    for _, row in sorted_df.iterrows():
        oracle_call = int(row['oracle_call'])
        score = float(row['score'])
        smiles = row['smiles']
        
        if oracle_call > max_oracle_calls:
            break
        
        # Determine the key for uniqueness
        if check_uniqueness:
            key = canonicalize_smiles(smiles)
            if key is None:
                continue
        else:
            key = smiles
        
        # Check if this molecule is new or has a better score
        old_score = unique_molecules.get(key)
        if old_score is not None and score <= old_score:
            # Not an improvement, just record current average
            top_n_avg_at_call[oracle_call] = top_n_sum / len(top_n_heap) if top_n_heap else 0.0
            continue
        
        # Update the molecule's best score
        unique_molecules[key] = score
        
        # Update the top-n heap
        if key in top_n_keys:
            # Molecule already in top-n, need to update its score
            # Remove old entry and re-add (heap doesn't support update)
            top_n_sum -= old_score
            top_n_heap = [(s, k) for s, k in top_n_heap if k != key]
            heapq.heapify(top_n_heap)
            heapq.heappush(top_n_heap, (score, key))
            top_n_sum += score
        elif len(top_n_heap) < top_n:
            # Heap not full yet, just add
            heapq.heappush(top_n_heap, (score, key))
            top_n_keys.add(key)
            top_n_sum += score
        elif score > top_n_heap[0][0]:
            # New score is better than the worst in top-n
            # Remove the worst, add the new one
            old_worst_score, old_worst_key = heapq.heappop(top_n_heap)
            top_n_keys.remove(old_worst_key)
            top_n_sum -= old_worst_score
            
            heapq.heappush(top_n_heap, (score, key))
            top_n_keys.add(key)
            top_n_sum += score
        
        # Record the current top-n average
        top_n_avg_at_call[oracle_call] = top_n_sum / len(top_n_heap) if top_n_heap else 0.0
    
    if not top_n_avg_at_call:
        return 0.0
    
    # Compute AUC using trapezoidal integration (same as PMO benchmark)
    oracle_calls = sorted(top_n_avg_at_call.keys())
    
    auc = 0.0
    prev_call = 0
    prev_avg = 0.0
    
    for call in oracle_calls:
        # Trapezoidal rule: area = width * (height1 + height2) / 2
        auc += (call - prev_call) * (top_n_avg_at_call[call] + prev_avg) / 2
        prev_call = call
        prev_avg = top_n_avg_at_call[call]
    
    # Extend to max_oracle_calls with the last value (plateau)
    auc += (max_oracle_calls - prev_call) * prev_avg
    
    return auc / max_oracle_calls


# Load data with SMILES
print("Loading our results with SMILES...")
our_df_smiles = load_our_results_with_smiles(OUR_RESULTS_DIR, OUR_MODEL_NAME)
print(f"Our results (with description): {len(our_df_smiles)} rows, {our_df_smiles['task'].nunique()} tasks")

our_df_smiles_no_desc = load_our_results_with_smiles(OUR_RESULTS_DIR_NO_DESC, OUR_MODEL_NAME_NO_DESC)
print(f"Our results (no description): {len(our_df_smiles_no_desc)} rows, {our_df_smiles_no_desc['task'].nunique()} tasks")

print("\nLoading PMO 10K results with SMILES...")
pmo_10k_df_smiles = load_pmo_10k_results_with_smiles(PMO_10K_RESULTS_DIR, MODEL_MAPPING_10K)
print(f"PMO 10K results: {len(pmo_10k_df_smiles)} rows, {pmo_10k_df_smiles['model'].nunique()} models")

# Combine dataframes
df_1000 = pd.concat([our_df_smiles, our_df_smiles_no_desc, pmo_10k_df_smiles], ignore_index=True)
print(f"\nCombined dataframe: {len(df_1000)} rows, {df_1000['model'].nunique()} models")

Loading our results with SMILES...
Our results (with description): 4121 rows, 28 tasks
Our results (no description): 3421 rows, 23 tasks

Loading PMO 10K results with SMILES...
PMO 10K results: 4848166 rows, 5 models

Combined dataframe: 4855708 rows, 7 models


In [33]:
# Compute AUC Top-10 for each model/task/run over 1000 iterations
MAX_ORACLE_CALLS_1000 = 1000

# Models that need uniqueness checking (our models use JSON files)
MODELS_CHECK_UNIQUENESS = {OUR_MODEL_NAME, OUR_MODEL_NAME_NO_DESC}

print("Computing AUC Top-10 over 1000 iterations...")
auc_top10_records = []
for (model, task, run), group in df_1000.groupby(['model', 'task', 'run']):
    # Only check uniqueness for our model; PMO baselines already have unique molecules
    check_uniqueness = model in MODELS_CHECK_UNIQUENESS
    auc = compute_auc_top10_from_df(group, MAX_ORACLE_CALLS_1000, check_uniqueness=check_uniqueness)
    auc_top10_records.append({
        'model': model,
        'task': task,
        'run': run,
        'auc_top10': auc
    })

auc_top10_df = pd.DataFrame(auc_top10_records)
print(f"Computed AUC Top-10 for {len(auc_top10_df)} model/task/run combinations")

# Compute mean and std AUC per model/task
auc_top10_summary = auc_top10_df.groupby(['model', 'task'])['auc_top10'].agg(['mean', 'std']).reset_index()
auc_top10_summary.columns = ['model', 'task', 'auc_mean', 'auc_std']

# Compute bootstrap 95% CI for sum across tasks
print("Computing bootstrap 95% CI for sums (Top-10 AUC, 1000 iterations)...")
model_sums_ci_top10 = bootstrap_sum_ci(auc_top10_df, 'auc_top10', n_bootstrap=10000, ci=0.95, seed=42)
model_sums_ci_top10 = model_sums_ci_top10.sort_values('sum_mean', ascending=False).reset_index(drop=True)
model_sums_ci_top10['rank'] = range(1, len(model_sums_ci_top10) + 1)

# Display with relative CI values
model_sums_ci_top10_display = model_sums_ci_top10.copy()
model_sums_ci_top10_display['ci_upper'] = model_sums_ci_top10_display['sum_ci_upper'] - model_sums_ci_top10_display['sum_mean']
model_sums_ci_top10_display['ci_lower'] = model_sums_ci_top10_display['sum_mean'] - model_sums_ci_top10_display['sum_ci_lower']
model_sums_ci_top10_display = model_sums_ci_top10_display[['model', 'sum_mean', 'ci_upper', 'ci_lower', 'rank']]

print(f"\nTop models by sum of AUC Top-10 (1000 iterations) with 95% CI:")
print(model_sums_ci_top10_display.to_string(index=False, formatters={
    'sum_mean': '{:.3f}'.format,
    'ci_upper': '+{:.3f}'.format,
    'ci_lower': '-{:.3f}'.format
}))

Computing AUC Top-10 over 1000 iterations...
Computed AUC Top-10 for 728 model/task/run combinations
Computing bootstrap 95% CI for sums (Top-10 AUC, 1000 iterations)...

Top models by sum of AUC Top-10 (1000 iterations) with 95% CI:
                     model sum_mean ci_upper ci_lower  rank
          LLM Agent (Ours)   20.240   +0.525   -0.480     1
LLM Agent (No Description)    9.906   +0.864   -1.042     2
                  Graph GA    8.478   +0.405   -0.397     3
                     GP BO    8.151   +0.398   -0.365     4
                  REINVENT    7.711   +0.317   -0.270     5
                    STONED    7.699   +0.304   -0.299     6
          REINVENT SELFIES    7.599   +0.223   -0.205     7


In [34]:
# Create comparison table for AUC Top-10 (1000 iterations)
model_order_top10 = model_sums_ci_top10['model'].tolist()
comparison_table_top10 = create_comparison_table(auc_top10_summary, model_sums_ci_top10, TASKS, model_order_top10)

print("AUC Top-10 over 1000 iterations (individual tasks):")
comparison_table_top10

AUC Top-10 over 1000 iterations (individual tasks):


,Task,LLM Agent (Ours),LLM Agent (No Description),Graph GA,GP BO,REINVENT,STONED,REINVENT SELFIES
0,albuterol_similarity,0.999± 0.000,0.495± 0.093,0.425± 0.022,0.514± 0.047,0.439± 0.016,0.434± 0.024,0.424± 0.013
1,amlodipine_mpo,0.898± 0.001,0.516± 0.027,0.498± 0.005,0.482± 0.013,0.472± 0.006,0.509± 0.008,0.464± 0.009
2,celecoxib_rediscovery,0.883± 0.002,0.422± 0.060,0.311± 0.017,0.359± 0.026,0.332± 0.018,0.294± 0.006,0.288± 0.009
3,deco_hop,0.956± 0.023,0.593± 0.003,0.576± 0.003,0.580± 0.006,0.574± 0.003,0.575± 0.001,0.565± 0.003
4,drd2,0.999± 0.000,0.652± 0.541,0.653± 0.129,0.384± 0.089,0.463± 0.076,0.274± 0.103,0.444± 0.063
5,fexofenadine_mpo,0.983± 0.003,0.595± 0.043,0.630± 0.010,0.606± 0.007,0.626± 0.013,0.633± 0.009,0.638± 0.006
6,gsk3b,0.974± 0.031,0.420± 0.130,0.374± 0.081,0.380± 0.064,0.338± 0.073,0.300± 0.035,0.261± 0.023
7,isomers_c7h8n2o2,0.999± 0.001,0.796± 0.112,0.334± 0.110,0.173± 0.095,0.088± 0.079,0.168± 0.077,0.195± 0.089
8,isomers_c9h10n2o2pf2cl,0.998± 0.000,0.443± 0.147,0.218± 0.092,0.105± 0.088,0.027± 0.018,0.135± 0.076,0.109± 0.022
9,jnk3,0.464± 0.065,0.134± 0.077,0.169± 0.034,0.179± 0.019,0.149± 0.007,0.179± 0.036,0.125± 0.022
